# Component 1 — Image Feature Extraction

This notebook runs a multimodal LLM (Claude) over each image in our dataset and extracts structured features.

Features extracted: number of people, age brackets, indoor/outdoor, color temperature, brightness, nature elements, time of day, and scene type.

In [1]:
import anthropic
import base64
import json
import os
import glob

client = anthropic.Anthropic()

IMAGE_DIR = "images"
OUTPUT_FILE = "image_features.json"

PROMPT = """Look at this image and return ONLY a JSON object (no other text) with these fields:

- num_people: integer
- age_brackets: list of strings, any of ["child", "adult", "older_adult"] present
- setting: "indoor" or "outdoor"
- color_temperature: "warm", "cool", or "neutral"
- brightness: "bright", "dim", or "moderate"
- nature_present: list of strings, any of ["water", "plants", "animals"] present, or empty list
- time_of_day: your best guess, or "unclear"
- scene_type: short description like "beach", "city street", "indoor portrait"

Return valid JSON only, no markdown formatting, no explanation."""

image_paths = sorted(
    glob.glob(os.path.join(IMAGE_DIR, "*.jpg"))
    + glob.glob(os.path.join(IMAGE_DIR, "*.JPG"))
    + glob.glob(os.path.join(IMAGE_DIR, "*.jpeg"))
    + glob.glob(os.path.join(IMAGE_DIR, "*.png"))
)

print(f"Found {len(image_paths)} images")

Found 15 images


In [2]:
# Run extraction on all images
# NOTE: This calls the Claude API once per image. Skip this cell if you already have image_features.json

results = {}

for path in image_paths:
    filename = os.path.basename(path)
    print(f"Analyzing {filename}...")

    with open(path, "rb") as f:
        image_data = base64.standard_b64encode(f.read()).decode("utf-8")

    ext = filename.rsplit(".", 1)[-1].lower()
    media_type = "image/png" if ext == "png" else "image/jpeg"

    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": image_data}},
                    {"type": "text", "text": PROMPT},
                ],
            }
        ],
    )

    raw = message.content[0].text
    try:
        features = json.loads(raw)
    except json.JSONDecodeError:
        print(f"  WARNING: couldn't parse JSON for {filename}")
        features = {"_raw": raw, "_parse_error": True}

    results[filename] = features

with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nDone — wrote features for {len(results)} images to {OUTPUT_FILE}")

Analyzing brooklyn.JPG...
Analyzing cat.png...
Analyzing children.JPG...
Analyzing cityview.JPG...
Analyzing dog.png...
Analyzing duck.png...
Analyzing family.JPG...
Analyzing fish.JPG...
Analyzing garden.JPG...
Analyzing newyork.JPG...
Analyzing ocean.JPG...
Analyzing race.png...
Analyzing rock.jpg...
Analyzing sanfran.JPG...
Analyzing waterfall.png...

Done — wrote features for 15 images to image_features.json


In [3]:
# Load and display the extracted features
with open("image_features.json") as f:
    image_features = json.load(f)

for name, features in image_features.items():
    print(f"\n--- {name} ---")
    print(json.dumps(features, indent=2))


--- brooklyn.JPG ---
{
  "num_people": 2,
  "age_brackets": [
    "adult"
  ],
  "setting": "outdoor",
  "color_temperature": "cool",
  "brightness": "moderate",
  "nature_present": [],
  "time_of_day": "daytime",
  "scene_type": "Brooklyn Bridge pedestrian walkway with NYC skyline"
}

--- cat.png ---
{
  "num_people": 0,
  "age_brackets": [],
  "setting": "indoor",
  "color_temperature": "warm",
  "brightness": "dim",
  "nature_present": [
    "animals"
  ],
  "time_of_day": "unclear",
  "scene_type": "bedroom with cat"
}

--- children.JPG ---
{
  "num_people": 3,
  "age_brackets": [
    "child",
    "adult"
  ],
  "setting": "outdoor",
  "color_temperature": "warm",
  "brightness": "bright",
  "nature_present": [
    "plants"
  ],
  "time_of_day": "daytime",
  "scene_type": "suburban plaza or campus courtyard"
}

--- cityview.JPG ---
{
  "num_people": 0,
  "age_brackets": [],
  "setting": "outdoor",
  "color_temperature": "cool",
  "brightness": "moderate",
  "nature_present": [
   

## Validation Against Manual Labels

To assess extraction quality, we manually labeled a subset of images and compared against Claude's output.

In [ ]:
# Manual labels for validation — fill these in based on your own observation of each image
manual_labels = {
    "garden.JPG": {"setting": "outdoor", "nature_present": ["plants"], "brightness": "bright"},
    "city_small.JPG": {"setting": "outdoor", "nature_present": [], "brightness": "bright"},
    "rock.jpg": {"setting": "outdoor", "nature_present": [], "brightness": "bright"},
    # Add more manual labels here...
}

print("Comparing manual labels vs Claude extraction:\n")
for name, manual in manual_labels.items():
    extracted = image_features.get(name, {})
    print(f"--- {name} ---")
    for key, manual_val in manual.items():
        extracted_val = extracted.get(key)
        match = "✓" if extracted_val == manual_val else "✗"
        print(f"  {key}: manual={manual_val}, claude={extracted_val}  {match}")
    print()